In [ ]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数

    评测时平台会自动替换 datasources / start_date / end_date 三个入参并调用本函数

    参数:
        datasources (dict): 数据源表名映射 {逻辑名: 物理表名}。一个因子可同时用到多张表，
                            通过逻辑名取出该阶段实际的物理表名，平台会在公榜/私榜自动切换。
                            当前可用逻辑名:
                                "bar1m"     -> 分钟 K 线表
        start_date (str): 开始时间
        end_date (str):   结束时间

    返回:
        pd.DataFrame: 因子数据，须包含三列 ['date', 'instrument', 'factor']，且不含 inf
    """
    import pandas as pd
    import dai

    # 从映射里取出本阶段实际的物理表名（切勿在 SQL 里硬编码表名，否则公榜/私榜无法切换）
    bar1m = datasources["bar1m"]

    # 若计算滚动/时序类因子，可以多取若干天数据作为缓冲（本示例无需滚动，仅作演示）
    LOOKBACK_DAYS = 7
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)

    # ===== 编写因子 SQL =====
    # 示例因子：基于 1 分钟盘口快照计算日内订单簿压力，按交易日聚合为日频因子
    # DAI 函数文档：https://bigquant.com/wiki/doc/Rceb2JQBdS
    sql = f"""
    WITH cte_snapshot AS (
        SELECT
            date, instrument_id,

            -- 交易日
            strftime(date, '%Y-%m-%d') as trading_day,

            -- 计算 加权5档买方量
            (
                COALESCE(bid_volume1, 0) * 1.0 + 
                COALESCE(bid_volume2, 0) * EXP(-0.3) + 
                COALESCE(bid_volume3, 0) * EXP(-0.6) + 
                COALESCE(bid_volume4, 0) * EXP(-0.9) + 
                COALESCE(bid_volume5, 0) * EXP(-1.2)
            ) as weight_bid,

            -- 计算 加权5档卖方量
            (
                COALESCE(ask_volume1, 0) * 1.0 + 
                COALESCE(ask_volume2, 0) * EXP(-0.3) + 
                COALESCE(ask_volume3, 0) * EXP(-0.6) + 
                COALESCE(ask_volume4, 0) * EXP(-0.9) + 
                COALESCE(ask_volume5, 0) * EXP(-1.2)
            ) as weight_ask,

            -- 计算 加权订单簿不平衡度
            (weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8) as weighted_imbalance,

            -- 按固定的每日时间截面分段；time_segment 只能取 _SECTION_TIMES 中的值
            CASE 
                -- 上午时间段
                WHEN strftime(date, '%H%M%S') > '090000' AND strftime(date, '%H%M%S') <= '093000' THEN 93000
                WHEN strftime(date, '%H%M%S') > '093000' AND strftime(date, '%H%M%S') <= '100000' THEN 100000
                WHEN strftime(date, '%H%M%S') > '100000' AND strftime(date, '%H%M%S') <= '103000' THEN 103000
                WHEN strftime(date, '%H%M%S') > '103000' AND strftime(date, '%H%M%S') <= '110000' THEN 110000
                WHEN strftime(date, '%H%M%S') > '110000' AND strftime(date, '%H%M%S') <= '113000' THEN 113000
                
                -- 下午时间段
                WHEN strftime(date, '%H%M%S') > '130000' AND strftime(date, '%H%M%S') <= '133000' THEN 133000
                WHEN strftime(date, '%H%M%S') > '133000' AND strftime(date, '%H%M%S') <= '140000' THEN 140000
                WHEN strftime(date, '%H%M%S') > '140000' AND strftime(date, '%H%M%S') <= '143000' THEN 143000
                
                -- 其他时间段（如果有数据）
                ELSE -1
            END as time_segment

        FROM {bar1m}
        WHERE time_segment != -1
    ),
    -- 用每个 30 分钟区间内的平均盘口不平衡度作为截面因子
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,
            avg(weighted_imbalance) as factor
        FROM cte_snapshot
        GROUP BY instrument_id, trading_day, time_segment
        ORDER BY instrument_id, time_segment
    )
    -- 映射instrument_id到instrument列
    SELECT
        -- 转换为固定时间截面的 date 列
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        -- 正值表示买盘更强，负值表示卖盘更强
        f.factor as factor
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    # ===== 调用dai计算因子 =====
    # compression=True 会把 instrument 列转为 category 类型，显著降低内存占用
    df = dai.query(sql, filters={'date': [start_date, end_date]}, compression=True).df()

    # ===== 对齐股票池 =====
    # 数据源保留了 2020 年至今所有成分股的数据以便计算时序因子，
    # 因此需与中证 1000 成分股做内连接，只保留当日属于成分股的标的
    # cpt_jyc_2026_instruments 已经收录了2020年以来的所有数据，不用替换
    stk_pool = dai.query(
        "SELECT date, instrument FROM cpt_jyc_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    result = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])

    return result


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {'bar1m': 'cpt_jyc_2026_stock_bar1m'}
    start_date = '2020-01-01 00:00:00'
    end_date = '2020-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 评估系统：show=True 表示画出评估图表
    result = M.jyc_eval._latest(
        factor_data=factor_data,
        show=True,
    )
